# 02 · The Allen ontology and `lut.py`

Here we explore the ontology tree that `to_allen` and `simplify` walk, look at
why some Allen IDs need a "16-bit compatible" remapping, and generate
Freeview-ready lookup tables with `lut.py`.

In [1]:
import sys, os
from pathlib import Path
sys.path.insert(0, ".")          # nb_helpers.py lives next to the notebooks
WORK = Path("_work/02"); WORK.mkdir(parents=True, exist_ok=True)   # scratch outputs of this notebook

import numpy as np
import pandas as pd
import nibabel as nib
import matplotlib.pyplot as plt
import nb_helpers as H
pd.set_option("display.max_rows", 60, "display.max_colwidth", 70)

In [2]:
from nextbrain_utils.io import load_ontology, load_json
from nextbrain_utils import lut as L
ont = load_ontology()

def walk(node, depth=0, parent=None):
    """Yield (node, depth, parent) depth-first. This is the same order as the library's _recurse."""
    yield node, depth, parent
    for c in node.get("children", []):
        yield from walk(c, depth + 1, node)

nodes = pd.DataFrame(
    [(n["id"], n["acronym"], n["name"], d, p["id"] if p else None, len(n.get("children", [])))
     for n, d, p in walk(ont)],
    columns=["id", "acronym", "name", "depth", "parent", "n_children"])
print(len(nodes), "nodes; max depth", nodes.depth.max())
nodes.head(10)

3317 nodes; max depth 14


,id,acronym,name,depth,parent,n_children
0,10153,NP,neural plate,0,NaN,1
1,10154,NT,neural tube,1,10153.0,2
2,10155,Br,brain,2,10154.0,3
3,10156,F,forebrain (prosencephalon),3,10155.0,5
4,10157,FGM,gray matter of forebrain,4,10156.0,3
5,10506,FTS,transient structures of forebrain,5,10157.0,16
6,11580,SG,subpial granular zone,6,10506.0,8
7,11581,fSG,SG in frontal neocortex,7,11580.0,6
8,11587,fSGfp,SG in frontal polar cortex,8,11581.0,0
9,11582,fSGdm,SG in dorsomedial frontal cortex,8,11581.0,0


In [3]:
def show_tree(node, max_depth=3, depth=0):
    print("  " * depth + f"{node['acronym']:>8s}  {node['id']:>10d}  {node['name']}")
    if depth < max_depth:
        for c in node.get("children", []):
            show_tree(c, max_depth, depth + 1)

show_tree(ont, max_depth=4)

      NP       10153  neural plate
        NT       10154  neural tube
          Br       10155  brain
             F       10156  forebrain (prosencephalon)
             FGM       10157  gray matter of forebrain
             FWM       10557  white matter of forebrain
              FV       10595  ventricles of forebrain
             FSS       10609  surface structures of forebrain
             fbv   266441685  blood vessels of forebrain
             M       10648  midbrain (mesencephalon)
             MGM       10649  gray matter of midbrain
             MWM       10650  white matter of midbrain
              MV       10651  ventricle of midbrain
             MSS       10652  surface structures of midbrain
             mbv   266441705  blood vessels of midbrain
             H       10653  hindbrain (rhombencephalon)
             HGM       10654  gray matter of the hindbrain
             HWM       10668  white matter of hindbrain
              HV       10669  ventricles of hindbrain
  

### Finding a node and its ancestry

`simplify` relies on this structure: every node "owns" all its descendants.

In [4]:
by_id = {n["id"]: n for n, _, _ in walk(ont)}
parent = {n["id"]: (p["id"] if p else None) for n, _, p in walk(ont)}

def lineage(node_id):
    out = []
    while node_id is not None:
        out.append(f"{by_id[node_id]['acronym']} ({node_id})")
        node_id = parent[node_id]
    return " <- ".join(out)

for acr in ["Pu", "THM", "AMY"]:
    nid = int(nodes.loc[nodes.acronym == acr, "id"].iloc[0])
    print(f"{acr:4s}", lineage(nid), "\n")

Pu   Pu (10338) <- STR (10333) <- BN (10332) <- CN (10331) <- Tel (10158) <- FGM (10157) <- F (10156) <- Br (10155) <- NT (10154) <- NP (10153) 

THM  THM (10390) <- Die (10389) <- FGM (10157) <- F (10156) <- Br (10155) <- NT (10154) <- NP (10153) 

AMY  AMY (10361) <- CN (10331) <- Tel (10158) <- FGM (10157) <- F (10156) <- Br (10155) <- NT (10154) <- NP (10153) 



In [5]:
# Subtree under the thalamus (THM): every one of these IDs would collapse to 10390 in `simplify -l THM`
thm = by_id[int(nodes.loc[nodes.acronym == "THM", "id"].iloc[0])]
print(sum(1 for _ in walk(thm)), "nodes under THM")
show_tree(thm, max_depth=1)

116 nodes under THM
     THM       10390  thalamus
       DTH       10391  dorsal thalamus
       ETH       10451  epithalamus
       VTH       10461  ventral thalamus
         R       10464  reticular nucleus of thalamus


### Two ontology files

* `AllenBrainOntologyDev.json`: the *developmental* ontology (ontology_id 11,
  root "neural plate"). **This is the only one the code uses.** It contains
  both the main Brodmann-style hierarchy and a separate gyral hierarchy under
  "cerebral gyri and lobules".
* `AllenBrainOntology.json`: the adult "guide" ontology (ontology_id 7, root
  "brain"). It is bundled but not used. The unused function
  `to_allen._fscortex_to_allen_guide` was written for it.

In [6]:
guide = load_json(Path(L.__file__).parent / "lut" / "AllenBrainOntology.json")
print("dev  :", ont["name"],   ont["ontology_id"],   sum(1 for _ in walk(ont)),   "nodes")
print("guide:", guide["name"], guide["ontology_id"], sum(1 for _ in walk(guide)), "nodes")
gyr = by_id[int(nodes.loc[nodes.name == "cerebral gyri and lobules", "id"].iloc[0])]
print("\n'cerebral gyri and lobules' is the root of the gyral hierarchy:", lineage(gyr["id"]))

dev  : neural plate 11 3317 nodes
guide: brain 7 1839 nodes

'cerebral gyri and lobules' is the root of the gyral hierarchy: CeG (12112) <- FSS (10609) <- F (10156) <- Br (10155) <- NT (10154) <- NP (10153)


## Why "16-bit compatible" labels?

Most Allen IDs have 5 digits (10153 to about 13000), but some are 9-digit
numbers. These do not fit in an `int16`, so Freeview cannot display them. The
`-s16`/`compat16bits` option strips their prefixes:

| Original range | Remapped to | Rule |
|---|---|---|
| 146035001 – 146035199 | 1 – 199 | `− 146035000` |
| 146034600 – 146034999 | 600 – 999 | `− 146034000` |
| 266441000 – 266441999 | 3000 – 3999 | `− (266440000 − 2000)` (avoids the DK range 1000–2999) |
| 267499000 – 267499999 | 9000 – 9999 | `− 267490000` |

Let us check whether these four rules cover every large ID:

In [7]:
big = nodes[nodes.id > 32767].copy()
big["prefix"] = big.id // 1000
print(len(big), "nodes have IDs > int16 max")
print(big.groupby("prefix").id.agg(["count", "min", "max"]))

323 nodes have IDs > int16 max
        count        min        max
prefix                             
146034     84  146034614  146034998
146035     23  146035004  146035124
266441    158  266441105  266441741
267498      6  267498979  267498999
267499     52  267499003  267499207


In [8]:
def s16(label):
    """Same arithmetic as lut.make_allen_lut / to_allen / simplify."""
    p = label // 1000
    if p == 146035: return label - 146035000
    if p == 146034: return label - 146034000
    if p == 266441: return label - (266440000 - 2000)
    if p == 267499: return label - 267490000
    return label

remapped = nodes.id.map(s16)
print("max after remap:", remapped.max(), "(fits int16:", remapped.max() < 32768, ")")
print("still unique   :", remapped.is_unique)
print("collides with DK 2000-2035:", bool(remapped.between(2000, 2035).any()))
print("\nstill > int16 after remap:")
print(nodes.loc[remapped > 32767, ["id", "acronym", "name"]].to_string(index=False))

max after remap: 267498999 (fits int16: False )
still unique   : True
collides with DK 2000-2035: False

still > int16 after remap:
       id acronym                                       name
267498979    A7ip intraparietal subdivision of area 7 (A7ip)
267498983   A7ipr             area 7ip, rostral part (A7ipr)
267498987  A7ipr1                       layer I of area 7ipr
267498991  A7ipr2                      layer II of area 7ipr
267498995  A7ipr3                     layer III of area 7ipr
267498999  A7ipr4                      layer IV of area 7ipr


### ⚠️ Issue 11: the prefix `267498` is forgotten

Six nodes (the area 7ip family) have IDs 267498979–267498999. The prefix is
`267498`, not `267499`, so none of the rules apply and they stay 9-digit.
NextBrain label 594 (`area_7ip__rostral_part_(a7ipr)`) maps to one of them.
`to_allen(..., compat16bits=True)` then calls `.astype('i2')`, which wraps
around **silently**:

In [9]:
print("267498983 as int16 ->", np.array([267498983]).astype("i2")[0])
# A natural fix would be to treat 267498 like 267499, e.g. 267498979 - 267489000 = 9979 is free:
print("9979 already used after remap?", bool((remapped == 9979).any()))

267498983 as int16 -> -18969
9979 already used after remap? False


## `lut.py`: generating FreeSurfer lookup tables

* `make_allen_lut(fname, acronym, compat16bits)` walks the tree depth-first and
  *yields* `(id, name, r, g, b, 0)`. Spaces in names become `_`, and colours
  come from `color_hex_triplet` via `_hex2rgb`.
* `make_dk_lut()` yields the DK cortical rows from `NextBrainLUT.txt`.
* `allen_lut(acronym, append_dk, compat16bits, save)` combines the two. It
  returns the rows as a list and, if `save` is set, writes them with
  `_line2str` (fixed-width columns). `save=True` writes `AllenBrainLUT.txt`
  in the current directory.

In [10]:
rows = L.allen_lut()
print(len(rows), "rows"); rows[:4]

3317 rows


[(10153, 'neural_plate', 215, 216, 216, 0),
 (10154, 'neural_tube', 208, 208, 209, 0),
 (10155, 'brain', 224, 224, 224, 0),
 (10156, 'forebrain_(prosencephalon)', 235, 214, 208, 0)]

In [11]:
print(L.allen_lut(acronym=True)[:3])
print(L._hex2rgb("EDA65F"), "|", L._line2str(10338, "putamen", 1, 2, 3, 0))

[(10153, 'NP', 215, 216, 216, 0), (10154, 'NT', 208, 208, 209, 0), (10155, 'Br', 224, 224, 224, 0)]
(237, 166, 95) |      10338  putamen                                                                   1   2   3   0


In [12]:
out = WORK / "AllenBrainLUT_s16.txt"
rows16 = L.allen_lut(append_dk=True, compat16bits=True, save=out)
print(open(out).read()[:400])
print("max id:", max(r[0] for r in rows16))

     10153  neural_plate                                                            215 216 216   0
     10154  neural_tube                                                             208 208 209   0
     10155  brain                                                                   224 224 224   0
     10156  forebrain_(prosencephalon)                                              235 214 208   0

max id: 267498999


### ⚠️ Bug: `append_dk=True` adds no DK rows

`make_dk_lut` only yields names that start with `ctx-rh-`. Since commit
`310dbf2 Align LUT`, `NextBrainLUT.txt` names them `ctx-<name>`, so the filter
matches nothing. The `allen+dk` LUT, which is also the CLI default, is
therefore missing the 2000–2035 rows, and DK voxels in an `allen -c dk` map
appear unlabelled in Freeview.

(There is a second, dormant bug: `if not f:` should be `if not line:`. A blank
line in the LUT would raise in `line.split()` unpacking.)

In [13]:
print("make_dk_lut() rows:", len(list(L.make_dk_lut())))
print("allen_lut(append_dk=True) - allen_lut():", len(L.allen_lut(append_dk=True)) - len(L.allen_lut()))

make_dk_lut() rows: 0


allen_lut(append_dk=True) - allen_lut(): 0


Until this is fixed, here is a workaround: build the DK rows yourself from
`NextBrainLUT.txt`.

In [14]:
from nextbrain_utils.io import load_lut
nxb = load_lut()
dk_rows = [(int(r["ID"]), r["NAME"].decode(), int(r["R"]), int(r["G"]), int(r["B"]), 0)
           for r in nxb if r["ID"] >= 2000]
allen_dk_rows = L.allen_lut(compat16bits=True) + dk_rows
with open(WORK / "AllenBrainLUT_s16_dk_fixed.txt", "w") as f:
    f.writelines(L._line2str(*r) + "\n" for r in allen_dk_rows)
print(len(dk_rows), "DK rows appended; total", len(allen_dk_rows))

36 DK rows appended; total 3353


Next: **03 · Combining hemispheres**.